# Channel predictions: `y_true` vs `y_pred`

Companion to `compare_scorer_fit_scope.ipynb`. That notebook is about *scores and
flags*; this one strips all of that away and just shows, per channel, what the
forecaster actually predicts versus what was observed.

Two views per panel:

1. **Full dataset** — one-step-ahead prediction on observed lags across every row,
   with the train / val / test boundaries marked. Residuals over the train split
   are in-sample (optimistically small); the gap between the curves should widen
   once you cross into val/test.
2. **Test set only** — the held-out tail (`>= test_start_timestamp`), zoomed in so
   the out-of-sample fit is actually legible.

Predictions come straight from `SpotforecastPredictor.predict` (the same adapter
the detector uses), so this reflects the real model on disk.

In [ ]:
from __future__ import annotations

from pathlib import Path

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd

from spotanomaly2.application.config import load_config
from spotanomaly2.application.data_manager import DataManager
from spotanomaly2.domain.anomaly_detector import AnomalyDetector
from spotanomaly2.domain.spotforecast_adapter import SpotforecastPredictor
from spotanomaly2.infrastructure import logging

# Point this at your runtime config (falls back to the bundled default).
CONFIG_PATH = Path("../config/config.yaml")
if not CONFIG_PATH.exists():
    CONFIG_PATH = Path("../config/default.yaml")

logger = logging.get_logger("channel_predictions")
config = load_config(CONFIG_PATH, base_dir=CONFIG_PATH.resolve().parent.parent)
print(f"Loaded config from {CONFIG_PATH}")

In [ ]:
# Load the processed panels. Needs a trained model on disk too — run
# `uv run spotanomaly2 all` (or process + train) first if model loading raises.
data_manager = DataManager(config, logger)
panel_data = data_manager.load_processed_data()
print(f"Loaded {len(panel_data)} panel(s): {list(panel_data)}")
for pid, df in panel_data.items():
    print(f"  panel {pid}: {len(df)} rows, {df.index.min()} -> {df.index.max()}")

In [ ]:
# Predict one step ahead on observed lags across the whole series, for every
# panel. We load whatever model is on disk (detect.model_timestamp in the config
# pins a specific one; otherwise the latest is used) so this matches what the
# detector would score. `predict` returns a (n_rows x n_channels) frame of y_pred
# aligned to the input index; we keep the aligned y_true alongside it.
detector = AnomalyDetector(config, logger)
predictor = SpotforecastPredictor(config, logger)


def predict_panel(panel_id: str):
    """Return (true_df, pred_df, model_data) over the whole panel."""
    df_full = panel_data[panel_id]
    model_data = detector.load_forecasting_model(panel_id)
    pred = predictor.predict(model_data, df_full)
    true = df_full.loc[pred.index, list(pred.columns)]
    return true, pred, model_data


predictions = {pid: predict_panel(pid) for pid in panel_data}
for pid, (true, pred, _) in predictions.items():
    print(f"panel {pid}: predicted {pred.shape[1]} channel(s) over {len(pred):,} rows")

In [ ]:
# Helpers shared by both views.
BOUNDARIES = [
    ("train_end_timestamp", "green", "train end"),
    ("val_end_timestamp", "orange", "val end"),
    ("test_start_timestamp", "red", "test start"),
]


def test_slice(true: pd.DataFrame, pred: pd.DataFrame, model_data: dict):
    """Slice (true, pred) to the held-out test window (>= test_start_timestamp).

    Falls back to the full frame if the model carries no test boundary (older
    artifacts). Timestamp tz is aligned to the data index for a safe compare.
    """
    ts = model_data.get("test_start_timestamp")
    if not ts or not isinstance(true.index, pd.DatetimeIndex):
        return true, pred
    cutoff = AnomalyDetector._align_timestamp_to_index(pd.Timestamp(ts), true)
    mask = true.index >= cutoff
    return true.loc[mask], pred.loc[mask]


def plot_channel_predictions(true, pred, model_data, panel_id, label, show_boundaries):
    """One interactive figure: one row per channel, observed vs predicted.

    Drag a region to zoom; x-axes are linked across channels (shared_xaxes), so
    zooming one channel zooms them all. Double-click resets; use the mode-bar
    (top-right) for pan / box-zoom.
    """
    channels = list(pred.columns)
    fig = make_subplots(
        rows=len(channels),
        cols=1,
        subplot_titles=channels,
        shared_xaxes=True,
        vertical_spacing=min(0.04, 0.5 / max(len(channels), 1)),
    )
    for r, col in enumerate(channels, start=1):
        fig.add_trace(
            go.Scatter(
                x=true.index, y=true[col].values, mode="lines",
                line=dict(color="black", width=0.8), opacity=0.55,
                name="observed", legendgroup="observed", showlegend=(r == 1),
                hovertemplate="%{x}<br>observed=%{y:.3f}<extra></extra>",
            ),
            row=r, col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=pred.index, y=pred[col].values, mode="lines",
                line=dict(color="#1f77b4", width=0.8),
                name="predicted", legendgroup="predicted", showlegend=(r == 1),
                hovertemplate="%{x}<br>predicted=%{y:.3f}<extra></extra>",
            ),
            row=r, col=1,
        )
        if show_boundaries:
            for key, c, lbl in BOUNDARIES:
                bts = model_data.get(key)
                if bts:
                    x = pd.Timestamp(bts)
                    fig.add_shape(
                        type="line", x0=x, x1=x, yref="y domain", y0=0, y1=1,
                        line=dict(color=c, dash="dot", width=1), row=r, col=1,
                    )
                    if r == 1:
                        fig.add_annotation(
                            x=x, yref="y domain", y=1.0, text=lbl, showarrow=False,
                            yanchor="bottom", font=dict(size=9, color=c), row=r, col=1,
                        )
        fig.update_yaxes(title_text=col, row=r, col=1)
    fig.update_xaxes(title_text="timestamp", row=len(channels), col=1)
    fig.update_layout(
        height=max(200 * len(channels), 300),
        width=1100,
        title_text=f"Panel {panel_id} — observed vs predicted ({label})",
        template="plotly_white",
    )
    fig.show()


## Full dataset

Observed (black) vs one-step-ahead predicted (blue) for every channel across the
whole series. Vertical lines mark where the forecaster's train split ends and the
val/test windows begin.

In [ ]:
for pid in panel_data:
    true, pred, model_data = predictions[pid]
    plot_channel_predictions(true, pred, model_data, pid, "full dataset", show_boundaries=True)

## Test set only

The held-out tail (`>= test_start_timestamp`), zoomed so the out-of-sample fit is
legible. This is the window the detector calibrates the `"unseen"` scorer on.

In [ ]:
for pid in panel_data:
    true, pred, model_data = predictions[pid]
    true_t, pred_t = test_slice(true, pred, model_data)
    print(f"panel {pid}: test window {len(pred_t):,} rows ({pred_t.index.min()} -> {pred_t.index.max()})")
    plot_channel_predictions(true_t, pred_t, model_data, pid, "test set", show_boundaries=False)

## Per-channel fit error

One-step-ahead RMSE / MAE per channel, over the full series and over the test
window. NaN predictions (missing forecaster, or rows the model could not score)
are dropped pairwise before computing the error and counted in `n_nan`.

In [ ]:
def channel_errors(true: pd.DataFrame, pred: pd.DataFrame, panel_id: str, window: str):
    rows = []
    for col in pred.columns:
        t = true[col].to_numpy(dtype=float)
        p = pred[col].to_numpy(dtype=float)
        finite = np.isfinite(t) & np.isfinite(p)
        n_nan = int((~finite).sum())
        if finite.any():
            err = t[finite] - p[finite]
            rmse = float(np.sqrt(np.mean(err**2)))
            mae = float(np.mean(np.abs(err)))
        else:
            rmse = mae = float("nan")
        rows.append(
            {
                "panel": panel_id,
                "window": window,
                "channel": col,
                "rmse": round(rmse, 4),
                "mae": round(mae, 4),
                "n": int(finite.sum()),
                "n_nan": n_nan,
            }
        )
    return rows


error_rows = []
for pid in panel_data:
    true, pred, model_data = predictions[pid]
    true_t, pred_t = test_slice(true, pred, model_data)
    error_rows += channel_errors(true, pred, pid, "full")
    error_rows += channel_errors(true_t, pred_t, pid, "test")

errors = pd.DataFrame(error_rows).set_index(["panel", "channel", "window"]).sort_index()
errors